# Lab 2 · Tracing your Agent — opencode go 版

这是教学视频 **Lab 2: Tracing your Agent** 的本地复刻版：模型从 OpenAI 官方换成 opencode go 订阅的 `ox-alpha-free`，其余 Phoenix tracing 部分与教程一一对应。

| 教程 | 本 Notebook |
|---|---|
| OpenAI 官方 API | opencode go 官方端点 `https://opencode.ai/zen/go/v1` |
| `gpt-4o-mini` | `ox-alpha-free` |
| `client.chat.completions.create`（Chat Completions） | `client.responses.create`（**Responses API**） |
| `helper.get_openai_api_key()` | macOS Keychain 中的 `opencode-go-api-key` |
| `helper.get_phoenix_endpoint()` | 仓库根目录 `.env.phoenix` 自动发现 |

**三个关键差异（提前踩坑）：**

1. ox-alpha-free 只暴露 **Responses API** 协议，没有 `/chat/completions`；Phoenix 的 `OpenAIInstrumentor` 对两种协议都能自动采集 span。
2. 该端点的 `input` **只接受 message-list 格式**，传纯字符串会报 `[1214] Input cannot be empty`（下面封装了 `user_msg()` 帮你规避）。
3. API key 存在 macOS Keychain（和 `codex-ox` 同一份），Notebook 运行时动态读取，不落盘、不进 git。

**前置条件：**

- 本地 Phoenix 已启动：`tests/scripts/start-phoenix-local.sh`，浏览器打开 <http://127.0.0.1:6006>
- Kernel 选择 `Python (AI Interviewer · Phoenix Lab)`（即 `ai_interviewer/.venv`，依赖已齐）

In [2]:
import json
import os
import subprocess
from pathlib import Path

from openai import OpenAI

# Phoenix OTel tracing
from phoenix.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor

/Users/junjielong/workspace/my_ai_interviewer/ai_interviewer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 初始化 opencode go 客户端

教程里是 `client = OpenAI(api_key=get_openai_api_key())`；这里多两样东西：

- `base_url` 指向 opencode go 官方端点；
- 一个 `user_msg()` 小工具函数，把普通字符串包装成该端点要求的 message-list 格式。

In [9]:
def get_opencode_go_api_key() -> str:
    """优先读环境变量 OPENCODE_GO_API_KEY；否则从 macOS Keychain 动态读取。"""
    key = os.environ.get("OPENCODE_GO_API_KEY")
    if key:
        return key
    user = os.environ.get("USER", "")
    result = subprocess.run(
        ["security", "find-generic-password", "-a", user, "-s", "opencode-go-api-key", "-w"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0 or not result.stdout.strip():
        raise RuntimeError(
            "未找到 opencode go API key：请 export OPENCODE_GO_API_KEY=<your key>，"
            "或确认 Keychain 中存在服务名 opencode-go-api-key"
        )
    return result.stdout.strip()


client = OpenAI(
    api_key=get_opencode_go_api_key(),
    base_url="https://opencode.ai/zen/go/v1",  # opencode go 官方端点（Responses API 协议）
)

MODEL = "ox-alpha-free"


def user_msg(text: str) -> list:
    """该端点的 input 必须是 message-list 格式；传纯字符串会被上游判为空输入。"""
    return [{"type": "message", "role": "user", "content": [{"type": "input_text", "text": text}]}]

## 接入 Phoenix

对应教程的 `phoenix as px` / `register` / `OpenAIInstrumentor` 部分：

1. 先把仓库根目录 `.env.phoenix` 里的本地 Endpoint 和项目名加载进环境变量（`register()` 靠它们发现本地 Phoenix，不上报云端）；
2. `register()` 返回 OTLP TracerProvider；
3. `OpenAIInstrumentor` 给 openai SDK 打补丁——之后每一次 `responses.create()` 都会自动产生带完整属性的 span，业务代码零侵入。

In [4]:
def load_phoenix_env() -> None:
    """加载仓库根目录 .env.phoenix（已存在的环境变量不覆盖）。"""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / ".env.phoenix"
        if candidate.exists():
            for line in candidate.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    name, _, value = line.partition("=")
                    os.environ.setdefault(name.strip(), value.strip())
            print(f"已加载配置: {candidate}")
            break


load_phoenix_env()

tracer_provider = register()  # 自动发现 PHOENIX_COLLECTOR_ENDPOINT / PHOENIX_PROJECT_NAME
OpenAIInstrumentor(tracer_provider=tracer_provider).instrument()
print(f"tracing 已开启 -> {os.environ.get('PHOENIX_COLLECTOR_ENDPOINT')}, 项目: {os.environ.get('PHOENIX_PROJECT_NAME')}")

已加载配置: /Users/junjielong/workspace/my_ai_interviewer/.env.phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: ai-interviewer-agent-eval
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: 127.0.0.1:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

tracing 已开启 -> http://127.0.0.1:6006, 项目: ai-interviewer-agent-eval


## 第 1 次 traced 调用（非流式）

和教程一样，从一次最简单的调用开始。运行后去 Phoenix UI 里应该能看到一条新 Trace：根 span 是 `responses - ox-alpha-free`，展开能看到完整的 prompt/response 文本和 token 用量。

In [5]:
resp = client.responses.create(
    model=MODEL,
    input=user_msg("用一句话向后端工程师解释什么是 LLM observability"),
)
print(resp.output_text)
print("\ntoken 用量:", resp.usage)

一句话：**就像你用 APM 和链路追踪监控微服务的延迟、错误率和依赖调用一样，LLM observability 是把这套思路用在 LLM 请求上——追踪每次调用的 prompt、响应、token 用量、成本、延迟以及输出质量（如幻觉、相关性），让你能定位线上问题、控制成本并持续评估模型效果。**

核心区别在于：传统可观测性只关心“系统是否正常”，而 LLM observability 还要回答“输出是否正确、是否有用”。

token 用量: ResponseUsage(input_tokens=98, input_tokens_details=InputTokensDetails(cache_write_tokens=None, cached_tokens=64), output_tokens=404, output_tokens_details=None, total_tokens=502)


## 流式调用

Streaming 下 Instrumentor 会把增量拼回完整文本后再写入 span，所以在 Phoenix 里看到的仍是完整对话。

In [6]:
stream = client.responses.create(
    model=MODEL,
    input=user_msg("列出你会考察候选人的两个工程能力，各配一句话理由"),
    stream=True,
)
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

我会考察这两个能力：

1. **调试与定位问题的能力** —— 工程师大部分时间在和“不按预期工作的系统”打交道，这最能看出一个人在信息不全时如何系统地推理。

2. **权衡取舍的设计思维** —— 真实问题没有标准答案，能否在性能、成本、可维护性之间做出合理取舍，是区分“会写代码”和“能做工程”的关键。

需要的话，我可以补充针对这两项的具体面试题设计。


## Mini Agent：工具调用循环

教程 Lab 2 的重点是 tracing **agent**。这里实现一个最小 tool-calling 循环：模型决定何时调 `get_interview_stats` 工具 → 本地执行工具并把结果以 `function_call_output` 回传 → 模型基于结果给出最终回答。

在 Phoenix 中这条 Trace 会呈现 `LLM → TOOL → LLM` 的层级，这正是排查 agent 行为时最常用的视角。（数据全部虚构，符合本目录的脱敏要求。）

In [7]:
MOCK_DB = {
    "Drake": {"sessions": 12, "avg_score": 82.5, "weak_topics": ["系统设计", "行为面试"]},
}

TOOLS = [
    {
        "type": "function",
        "name": "get_interview_stats",
        "description": "查询指定候选人的模拟面试统计",
        "parameters": {
            "type": "object",
            "properties": {"candidate": {"type": "string", "description": "候选人代号"}},
            "required": ["candidate"],
        },
    }
]


def execute_tool(call) -> str:
    args = json.loads(call.arguments)
    return json.dumps(MOCK_DB.get(args.get("candidate", ""), {}), ensure_ascii=False)


def run_agent(question: str, max_rounds: int = 3) -> str:
    conversation = [{"type": "message", "role": "user", "content": [{"type": "input_text", "text": question}]}]
    for round_no in range(max_rounds):
        resp = client.responses.create(model=MODEL, input=conversation, tools=TOOLS)
        calls = [item for item in resp.output if item.type == "function_call"]
        if not calls:
            return resp.output_text
        # 把模型的 function_call 输出原样回传，再附上工具结果
        conversation.extend(item.model_dump(exclude_none=True) for item in resp.output)
        for call in calls:
            conversation.append(
                {"type": "function_call_output", "call_id": call.call_id, "output": execute_tool(call)}
            )
    return "达到最大轮数仍未得到最终回答"


print(run_agent("查一下 Drake 的模拟面试统计，指出他最需要补强的方向"))

已查询到 Drake 的模拟面试统计，情况如下：

## 📊 面试数据总览

| 指标 | 数值 |
|------|------|
| 模拟面试场次 | **12 次** |
| 平均得分 | **82.5 分** |

## ⚠️ 最需要补强的方向

根据数据分析，Drake 的薄弱环节集中在两个领域：

1. **系统设计** —— 这是他最需要优先补强的方向。系统设计通常考察架构思维、可扩展性、高并发处理等能力，建议通过大量练习经典题目（如设计短链接服务、消息队列、分布式缓存等）来积累。

2. **行为面试** —— 第二个薄弱点。可以通过 STAR 法则（情境-任务-行动-结果）整理个人项目经历和过往案例，多做模拟问答来提升表达的结构性和说服力。

**总体来看**：82.5 的平均分说明基础扎实，12 场的练习量也足够，只要针对性攻克这两个方向，整体表现应该会有明显提升。建议后续模拟面试中加大这两个题型的比重，并跟踪分数变化验证补强效果。


## 去 Phoenix 里核对

浏览器打开 <http://127.0.0.1:6006> → 项目 `ai-interviewer-agent-eval`，逐条检查：

1. **根 Trace 数量**：上面每次 `create()` 各一条；
2. **Span Attributes**：`llm.input_messages` / `llm.output_messages`（完整 prompt 与回复）、`llm.model_name`、`llm.token_count.prompt/completion/total`；
3. **Agent Trace**：mini agent 那条的层级应为 LLM（发起 function call）→ TOOL（get_interview_stats）→ LLM（最终总结）；
4. **横向对比**：同一问题多次运行的 latency 和 token 差异——这就是后续固定 Case、做评测基线的入口。

In [8]:
print(
    "Phoenix UI: "
    f"{os.environ.get('PHOENIX_COLLECTOR_ENDPOINT', 'http://127.0.0.1:6006')}"
    f"  (项目: {os.environ.get('PHOENIX_PROJECT_NAME', 'ai-interviewer-agent-eval')})"
)

Phoenix UI: http://127.0.0.1:6006  (项目: ai-interviewer-agent-eval)
